## Predikcija sekundarne strukture proteina primenom CNN+LSTM arhitekture i evolutivnih metoda optimizacije
-------
#### Sara Stojkov SV38/2023

Skup podataka koji će biti korišćen nalazi se na sledećem linku:
- CB513: https://www.kaggle.com/datasets/moklesur/cb513-dataset-for-protein-structure-prediction
- PSS (Protein Secondary Structure): https://www.kaggle.com/datasets/kirkdco/protein-secondary-structure-2022/data

CB513 dataset je u specifičnom .npy obliku - binarnom fajlu koji generiše numpy biblioteka i koji služi za čuvanje velikih array struktura podataka.
PSS je u csv formatu, pogodan za obradu.

Detaljniji opis strukture dataseta može se naći u ovom repozitorijumu: https://github.com/taneishi/CB513_dataset 


In [ ]:
# ovo pokrenuti samo jednom, služi da prebaci dataset u format za python 3

import numpy as np

data = np.load('data/CB513.npy', allow_pickle=True, encoding='latin1')

np.save('data/CB513.npy', data)

print("File has been successfully converted to the correct format for python 3")
# if no warning appears in the following cells, it means the conversion was successful

### 1. Upoznavanje sa podacima / Eksplorativna analiza i vizuelizacija

In [ ]:
from helpers.data_utils import load_csv_dataset

df = load_csv_dataset("data/2022-08-06-pdb-intersect-pisces_pc25_r2.5.csv")

print("Number of sequences:", len(df))
print(df.head(3))
print("\nExample sequence:", df.iloc[0]["seq"][:50])
print("Example sst3:     ", df.iloc[0]["sst3"][:50])
print("\nStatistics of sequence lengths:")
print(df["len"].describe())

In [ ]:
from sklearn.model_selection import train_test_split
from helpers.data_utils import dataframe_to_tensors

MAX_LEN = 400  # prilagodi na osnovu histograma iz prethodne ćelije

train_df, val_df = train_test_split(df, test_size=0.05, random_state=42)

X_train, y_train, mask_train = dataframe_to_tensors(train_df, MAX_LEN)
X_val, y_val, mask_val = dataframe_to_tensors(val_df, MAX_LEN)

print("Train:", X_train.shape, y_train.shape, mask_train.shape)
print("Val:  ", X_val.shape, y_val.shape, mask_val.shape)

# provera da maska ima smisla
print("Average percentage of 'real' positions (train):", mask_train.mean())

In [ ]:
data = np.load("data/CB513.npy")
print("Shape:", data.shape)
print("Min/Max values:", data.min(), data.max())
print("Average values of the first position of the first protein:")
print(data[0, 0, :30].mean() if data.ndim == 3 else data[0, :30].mean())

In [ ]:
import matplotlib.pyplot as plt

all_labels = y_train[mask_train == 1]
unique, counts = np.unique(all_labels, return_counts=True)
labels_names = ["H", "E", "C"]

plt.bar([labels_names[i] for i in unique], counts)
plt.title("Distribution of secondary structure classes (train)")
plt.ylabel("Number of residues")
plt.show()

print(dict(zip([labels_names[i] for i in unique], counts)))

### 2. model


In [ ]:
from helpers.model import CNNLSTMModel
import torch

m = CNNLSTMModel()
test_input = torch.randn(4, 400, 21)  # batch=4, seq_len=400, input_dim=21
out = m(test_input)
print(out.shape)  # očekuješ (4, 400, 3)

In [ ]:
from helpers.train import train_model

test_hp = {"n_filters": 32, "kernel_size": 5, "lstm_units": 64,
           "dropout": 0.3, "lr": 0.001, "batch_size": 32}

acc = train_model(test_hp, X_train, y_train, mask_train,
                   X_val, y_val, mask_val, max_epochs=2)
print("Val Q3 accuracy:", acc)